# 🔍 What does a neural network *see*?

<a href="https://colab.research.google.com/github/RCSchmelzle/RCSchmelzle.github.io/blob/main/comp380-inside-a-network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Train a tiny image classifier on real doodles, then open it up and look at what each layer does with a drawing.

**How to use:** click **Runtime → Run all**. No GPU needed; it takes about two minutes. Then scroll down, read, and play. In part 4 you draw.

| Part | What happens |
|---|---|
| 1 · Get the data | 25,000 doodles from Google's *Quick, Draw!* game |
| 2 · Train | the forward pass → loss → backprop → update loop, in real code |
| 3 · Look inside | the filters it learned, and what each layer does to a drawing |
| 4 · Draw your own | your doodle, layer by layer |
| 5 · How layers bend space | a network untangling two groups of points |
| 6 · When it goes wrong | skewed data, things it has never seen, memorizing |

In [ ]:
#@title ⚙️ Setup: run this first (the code is hidden; double-click to see it)
import io, base64, struct, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Javascript
from PIL import Image

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
try:
    from google.colab import output as colab_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CLASSES = ['cat', 'fish', 'house', 'tree', 'bicycle']
BLUE, ORANGE, RED, GREY = '#2f8fdd', '#e8590c', '#c1121f', '#8b929c'
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

# ---------------------------------------------------------------- data
URL = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy'

def load_doodles(name, n):
    """Download just the first n drawings of one Quick, Draw! category: 28x28 images, white strokes on black."""
    get = lambda lo, hi: urllib.request.urlopen(urllib.request.Request(
        URL.format(name), headers={'Range': f'bytes={lo}-{hi}'})).read()
    head = get(0, 15)
    start = 10 + struct.unpack('<H', head[8:10])[0] if head[6] == 1 else 12 + struct.unpack('<I', head[8:12])[0]
    return np.frombuffer(get(start, start + n * 784 - 1), dtype=np.uint8).reshape(-1, 28, 28)

class Split:
    def __init__(self, X, y): self.X, self.y = X, y

class Data:
    def __init__(self, train, test): self.train, self.test = train, test

_cache = {}
def get_data(n_train=4000, n_test=1000, keep=None):
    """Train/test split per class. keep={'fish': 0.05} keeps only 5% of that class's training drawings."""
    keep = keep or {}
    Xtr, ytr, Xte, yte = [], [], [], []
    for c, name in enumerate(CLASSES):
        if name not in _cache:
            _cache[name] = load_doodles(name, n_train + n_test)
        imgs = _cache[name]
        k = int(n_train * keep.get(name, 1.0))
        Xtr.append(imgs[:k]); ytr += [c] * k
        Xte.append(imgs[n_train:n_train + n_test]); yte += [c] * n_test
    t = lambda parts: torch.tensor(np.concatenate(parts), dtype=torch.float32).unsqueeze(1) / 255
    return Data(Split(t(Xtr), torch.tensor(ytr)), Split(t(Xte), torch.tensor(yte)))

_order = torch.Generator().manual_seed(SEED)
def batches(split, size=128):
    order = torch.randperm(len(split.y), generator=_order)
    for i in range(0, len(order), size):
        idx = order[i:i + size]
        yield split.X[idx], split.y[idx]

def _clean(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def show_examples(data, per_class=10):
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(per_class * 0.85, len(CLASSES) * 0.9))
    for c, name in enumerate(CLASSES):
        idx = (data.train.y == c).nonzero().flatten()[:per_class]
        for j, i in enumerate(idx):
            axes[c, j].imshow(data.train.X[i, 0], cmap='gray'); _clean(axes[c, j])
        axes[c, 0].set_ylabel(name, rotation=0, ha='right', va='center', fontsize=12)
    fig.suptitle('Some training doodles', fontsize=13)
    plt.show()

def show_as_numbers(data, index=0):
    img = (data.train.X[index, 0] * 255).round().int().numpy()
    r0, c0 = 6, 6
    patch = img[r0:r0 + 12, c0:c0 + 12]
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios': [1, 1.6]})
    a.imshow(img, cmap='gray'); _clean(a)
    a.add_patch(Rectangle((c0 - .5, r0 - .5), 12, 12, fill=False, edgecolor=RED, lw=2))
    a.set_title('What you see', fontsize=12)
    b.imshow(patch, cmap='gray', vmin=0, vmax=255); _clean(b)
    for (r, c), v in np.ndenumerate(patch):
        b.text(c, r, v, ha='center', va='center', fontsize=7.5, color='black' if v > 120 else 'white')
    b.set_title('What the network sees (the red square, as numbers)', fontsize=12)
    plt.show()

# ---------------------------------------------------------------- the model
class TinyCNN(nn.Module):
    def __init__(self, n_classes=len(CLASSES)):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)     # 8 small filters look at the raw pixels
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)    # 16 filters look at layer 1's output
        self.fc = nn.Linear(16 * 7 * 7, n_classes)     # one score per class

    def forward(self, x, keep=False):
        a1 = F.relu(self.conv1(x))
        a2 = F.relu(self.conv2(F.max_pool2d(a1, 2)))
        logits = self.fc(F.max_pool2d(a2, 2).flatten(1))
        return (logits, {'layer1': a1, 'layer2': a2}) if keep else logits

def describe(model):
    rows = [('Layer 1', '8 filters, each 3×3', model.conv1),
            ('Layer 2', '16 filters, each 3×3 across 8 channels', model.conv2),
            ('Output', f'every layer-2 value → {len(CLASSES)} class scores', model.fc)]
    total = 0
    for name, what, layer in rows:
        n = sum(p.numel() for p in layer.parameters()); total += n
        print(f'{name:8} {what:42} {n:6,} weights')
    print(f'{"":8} {"total":42} {total:6,} weights   (large language models have billions)')

@torch.no_grad()
def predict(model, X):
    return torch.cat([model(X[i:i + 1000]) for i in range(0, len(X), 1000)]).softmax(1)

def accuracy(model, split):
    return (predict(model, split.X).argmax(1) == split.y).float().mean().item()

def report(model, data):
    print(f'Correct on drawings it trained on:   {accuracy(model, data.train):6.1%}')
    print(f'Correct on held-out drawings:        {accuracy(model, data.test):6.1%}   ← the number that matters')

class LossPlot:
    """Live loss curve while training."""
    def __init__(self, every=20):
        self.losses, self.every = [], every
        self.fig, self.ax = plt.subplots(figsize=(8, 3))
        plt.close(self.fig)
        self.handle = display(self.fig, display_id=True)
    def add(self, value):
        self.losses.append(value)
        if len(self.losses) % self.every == 0: self._draw()
    def _draw(self):
        ax, L = self.ax, np.array(self.losses)
        ax.clear()
        ax.plot(L, color=BLUE, lw=1, alpha=.3)
        if len(L) > 20:
            ax.plot(np.arange(19, len(L)), np.convolve(L, np.ones(20) / 20, 'valid'), color=BLUE, lw=2.5)
        ax.set_xlabel('training step (each step = one batch of 128 doodles)')
        ax.set_ylabel('loss')
        ax.set_title(f'Loss: how wrong the network is   (now {L[-20:].mean():.2f})', fontsize=12)
        ax.set_ylim(0, max(1.8, L.max() * 1.05))
        self.handle.update(self.fig)
    def done(self):
        self._draw()

def quick_train(data, epochs=3, seed=SEED, lr=0.003):
    torch.manual_seed(seed)
    model = TinyCNN()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for images, labels in batches(data.train):
            loss = F.cross_entropy(model(images), labels)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

# ---------------------------------------------------------------- looking inside
def show_filters(model):
    w = model.conv1.weight.detach()[:, 0]
    lim = w.abs().max()
    fig, axes = plt.subplots(1, 8, figsize=(11, 1.9))
    for k, ax in enumerate(axes):
        ax.imshow(w[k], cmap='RdBu_r', vmin=-lim, vmax=lim); _clean(ax)
        ax.set_title(f'filter {k + 1}', fontsize=10)
    fig.suptitle('Layer 1 filters (red = positive weight, blue = negative)', fontsize=12, y=1.08)
    plt.show()

def _as_input(img):
    return torch.as_tensor(np.asarray(img), dtype=torch.float32).reshape(1, 1, 28, 28)

def show_layers(model, img):
    x = _as_input(img)
    with torch.no_grad():
        logits, acts = model(x, keep=True)
    probs = logits.softmax(1)[0].numpy()

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(4, 9, height_ratios=[1.7, 1, 1, 1], width_ratios=[1.3] + [1] * 8, hspace=.5, wspace=.15)
    ax = fig.add_subplot(gs[0, 1:3]); ax.imshow(x[0, 0], cmap='gray'); _clean(ax)
    ax.set_title('Input: 784 numbers', fontsize=11)

    axp = fig.add_subplot(gs[0, 4:9])
    order = np.argsort(probs)
    colors = [BLUE if i == order[-1] else GREY for i in order]
    axp.barh([CLASSES[i] for i in order], probs[order], color=colors)
    for yy, i in enumerate(order):
        axp.text(probs[i] + .01, yy, f'{probs[i]:.0%}', va='center', fontsize=10)
    axp.set_xlim(0, 1.12); axp.set_xticks([])
    axp.set_title(f'Output: “{CLASSES[order[-1]]}”', fontsize=12)

    for row, (key, n, label) in enumerate([('layer1', 8, 'Layer 1\n8 feature maps\n28×28'),
                                           ('layer2', 16, 'Layer 2\n16 feature maps\n14×14')]):
        a = acts[key][0]
        vmax = max(a.max().item(), 1e-6)
        for k in range(n):
            r = 1 + row + (k // 8 if key == 'layer2' else 0)
            axk = fig.add_subplot(gs[r, 1 + k % 8])
            axk.imshow(a[k], cmap='magma', vmin=0, vmax=vmax); _clean(axk)
        lab = fig.add_subplot(gs[1 + row, 0]); lab.axis('off')
        lab.text(1, .5 if key == 'layer1' else -.1, label, ha='right', va='center', fontsize=10.5)
    fig.text(.5, .015, 'Bright = this filter found its pattern here.  Dark = nothing here for it.',
             ha='center', fontsize=10, color=GREY)
    plt.show()

def pick(data, name, nth=0):
    """The nth held-out drawing of a class."""
    return data.test.X[(data.test.y == CLASSES.index(name)).nonzero().flatten()[nth]]

@torch.no_grad()
def show_channel_favorites(model, data, channels=8, k=6):
    X, y = data.test.X, data.test.y
    A = torch.cat([model(X[i:i + 1000], keep=True)[1]['layer2'] for i in range(0, len(X), 1000)])
    peak, where = A.flatten(2).max(2)                  # strongest response per drawing, per channel
    chosen = peak.mean(0).argsort(descending=True)[:channels]
    fig, axes = plt.subplots(channels, k, figsize=(k * 1.25, channels * 1.35))
    for r, ch in enumerate(chosen):
        for j, i in enumerate(peak[:, ch].topk(k).indices):
            ax = axes[r, j]; ax.imshow(X[i, 0], cmap='gray'); _clean(ax)
            yy, xx = divmod(where[i, ch].item(), 14)
            ax.add_patch(Rectangle((xx * 2 - 4, yy * 2 - 4), 9, 9, fill=False, edgecolor=RED, lw=1.8))
            ax.set_xlabel(CLASSES[y[i]], fontsize=8, labelpad=1)
        axes[r, 0].set_ylabel(f'channel {ch + 1}', rotation=0, ha='right', va='center', fontsize=10)
    fig.suptitle('Layer 2: the held-out drawings each channel responds to most\n(red box = where it responded)', fontsize=11)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------- drawing (Colab only)
_CANVAS_JS = r"""
async function getDrawing() {
  const wrap = document.createElement('div');
  wrap.style.fontFamily = 'sans-serif';
  const hint = document.createElement('div');
  hint.textContent = 'Draw ONE thing, as big as the box. Then click Done.';
  const c = document.createElement('canvas');
  c.width = c.height = 280;
  c.style.cssText = 'background:#000;border-radius:10px;touch-action:none;cursor:crosshair;display:block;margin:8px 0';
  const clear = document.createElement('button'); clear.textContent = 'Clear';
  const done = document.createElement('button'); done.textContent = 'Done ✓'; done.style.marginLeft = '8px';
  wrap.append(hint, c, clear, done);
  document.body.appendChild(wrap);
  const g = c.getContext('2d');
  const wipe = () => { g.fillStyle = '#000'; g.fillRect(0, 0, 280, 280); };
  wipe();
  g.strokeStyle = '#fff'; g.lineWidth = 16; g.lineCap = 'round'; g.lineJoin = 'round';
  let down = false;
  const pt = e => { const r = c.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; };
  c.addEventListener('pointerdown', e => { down = true; c.setPointerCapture(e.pointerId);
    g.beginPath(); g.moveTo(...pt(e)); g.lineTo(...pt(e)); g.stroke(); });
  c.addEventListener('pointermove', e => { if (down) { g.lineTo(...pt(e)); g.stroke(); } });
  c.addEventListener('pointerup', () => { down = false; });
  clear.onclick = wipe;
  await new Promise(resolve => { done.onclick = resolve; });
  const url = c.toDataURL('image/png');
  wrap.remove();
  return url;
}
"""

def to_28x28(png_bytes):
    """Crop to the drawing, scale its longest side to 26 px, center it: the same framing Quick, Draw! uses."""
    im = np.asarray(Image.open(io.BytesIO(png_bytes)).convert('L'), dtype=np.uint8)
    ys, xs = np.nonzero(im > 30)
    if len(xs) == 0:
        return np.zeros((28, 28), np.float32)
    im = im[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = im.shape
    scale = 26 / max(h, w)
    small = np.asarray(Image.fromarray(im).resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BOX))
    out = np.zeros((28, 28), np.float32)
    top, left = (28 - small.shape[0]) // 2, (28 - small.shape[1]) // 2
    out[top:top + small.shape[0], left:left + small.shape[1]] = small / 255
    return out

def draw():
    if not IN_COLAB:
        d = globals()['data']
        i = np.random.randint(len(d.test.y))
        print(f'(Drawing only works in Google Colab. Using a random held-out {CLASSES[d.test.y[i]]} instead.)')
        return d.test.X[i, 0].numpy()
    display(Javascript(_CANVAS_JS))
    url = colab_output.eval_js('getDrawing()')
    return to_28x28(base64.b64decode(url.split(',', 1)[1]))

# ---------------------------------------------------------------- part 5: bending space
def bend_space_demo(steps=3000, seed=1):
    rng = np.random.default_rng(seed)
    n = 300
    t = rng.uniform(0, np.pi, n)
    X = np.r_[np.c_[np.cos(t), np.sin(t)], np.c_[1 - np.cos(t), 0.45 - np.sin(t)]] + rng.normal(0, .07, (2 * n, 2))
    X = (X - X.mean(0)) / X.std(0)
    y = np.r_[np.zeros(n), np.ones(n)]
    Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

    torch.manual_seed(seed)
    layers = nn.ModuleList([nn.Linear(2, 2) for _ in range(3)])   # every hidden layer has just 2 neurons,
    head = nn.Linear(2, 1)                                         # so we can plot it on flat paper
    def run(x):
        out = [x]
        for L in layers:
            x = torch.tanh(L(x)); out.append(x)
        return out, head(x).squeeze(1)
    opt = torch.optim.Adam(list(layers.parameters()) + list(head.parameters()), lr=0.03)
    for _ in range(steps):
        _, logit = run(Xt)
        loss = F.binary_cross_entropy_with_logits(logit, yt)
        opt.zero_grad(); loss.backward(); opt.step()

    # a grid of lines to show how each layer stretches and folds the plane
    g = np.linspace(-2.6, 2.6, 23)
    lines = [np.c_[np.full(200, v), np.linspace(-2.6, 2.6, 200)] for v in g] + \
            [np.c_[np.linspace(-2.6, 2.6, 200), np.full(200, v)] for v in g]
    with torch.no_grad():
        pts, logit = run(Xt)
        grid = [run(torch.tensor(l, dtype=torch.float32))[0] for l in lines]
    acc = ((logit > 0).float() == yt).float().mean().item()

    titles = ['Input: two tangled groups', 'After layer 1', 'After layer 2', 'After layer 3: pulled apart,\none straight line splits them']
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for s, ax in enumerate(axes):
        for l in grid:
            p = l[s].numpy(); ax.plot(p[:, 0], p[:, 1], color='#d0d3d8', lw=.6, zorder=0)
        p = pts[s].numpy()
        ax.scatter(p[:, 0], p[:, 1], c=np.where(y == 0, BLUE, ORANGE), s=9, zorder=2)
        ax.set_title(titles[s], fontsize=11); _clean(ax)
        lo, hi = p.min(0), p.max(0); pad = (hi - lo).max() * .12 + .05; c = (lo + hi) / 2; half = (hi - lo).max() / 2 + pad
        ax.set_xlim(c[0] - half, c[0] + half); ax.set_ylim(c[1] - half, c[1] + half); ax.set_aspect('equal', adjustable='box')
    w, b = head.weight.detach()[0].numpy(), head.bias.item()
    xs = np.linspace(-1.05, 1.05, 50)
    if abs(w[1]) > 1e-6:
        axes[3].plot(xs, -(w[0] * xs + b) / w[1], color=RED, lw=2.5, zorder=3)
    plt.show()
    print(f'Points classified correctly: {acc:.0%}. The network never drew a curve. Each layer stretched and folded '
          f'the space until one straight cut was enough.')

# ---------------------------------------------------------------- part 6: when it goes wrong
def skewed_data_demo(model, data, rare='fish', keep=0.05):
    print(f'Retraining from scratch, but with only {keep:.0%} of the {rare} drawings '
          f'({int(4000 * keep)} instead of 4,000)…')
    skewed = quick_train(get_data(keep={rare: keep}))
    per_class = lambda m: [(predict(m, data.test.X).argmax(1)[data.test.y == c] == c).float().mean().item()
                           for c in range(len(CLASSES))]
    a, b = per_class(model), per_class(skewed)
    x = np.arange(len(CLASSES))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.bar(x - .2, a, .4, color=BLUE, label='balanced training data')
    ax.bar(x + .2, b, .4, color=ORANGE, label=f'only {keep:.0%} of the {rare} drawings')
    ax.set_xticks(x, CLASSES); ax.set_ylim(0, 1.05); ax.set_ylabel('correct on held-out drawings')
    ax.legend(loc='lower left', frameon=False)
    plt.show()
    i = CLASSES.index(rare)
    print(f'{rare}: {a[i]:.0%} → {b[i]:.0%}. Same network, same code. Only the training data changed.')

@torch.no_grad()
def never_seen_demo(model, name='banana', n=400):
    imgs = torch.tensor(load_doodles(name, n), dtype=torch.float32).unsqueeze(1) / 255
    p = predict(model, imgs)
    conf, cls = p.max(1)
    fig = plt.figure(figsize=(13, 3.4))
    gs = fig.add_gridspec(1, 9, width_ratios=[1] * 6 + [.3, 2.4, .2])
    for j in range(6):
        ax = fig.add_subplot(gs[0, j]); ax.imshow(imgs[j, 0], cmap='gray'); _clean(ax)
        ax.set_title(f'“{CLASSES[cls[j]]}”\n{conf[j]:.0%} sure', fontsize=10)
    ax = fig.add_subplot(gs[0, 7])
    counts = np.bincount(cls.numpy(), minlength=len(CLASSES)) / n
    ax.barh(CLASSES, counts, color=GREY); ax.set_xlim(0, 1)
    ax.set_title(f'What it called {n} {name}s', fontsize=11)
    plt.show()
    print(f'The network has never seen a {name}, and "{name}" is not one of its answers. It still answered, '
          f'with an average confidence of {conf.mean():.0%}.')

def memorize_demo(data, per_class=40, epochs=120):
    idx = torch.cat([(data.train.y == c).nonzero().flatten()[:per_class] for c in range(len(CLASSES))])
    tiny = Data(Split(data.train.X[idx], data.train.y[idx]), data.test)
    torch.manual_seed(SEED)
    model = TinyCNN()
    opt = torch.optim.Adam(model.parameters(), lr=0.003)
    hist = []
    for ep in range(epochs):
        for images, labels in batches(tiny.train, size=50):
            loss = F.cross_entropy(model(images), labels)
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 5 == 4:
            hist.append((ep + 1, accuracy(model, tiny.train), accuracy(model, tiny.test)))
    e, tr, te = map(np.array, zip(*hist))
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(e, tr, color=ORANGE, lw=2.5, label=f'the {per_class * len(CLASSES)} drawings it trained on')
    ax.plot(e, te, color=BLUE, lw=2.5, label='5,000 held-out drawings')
    ax.set_ylim(0, 1.05); ax.set_xlabel('passes over the training drawings'); ax.set_ylabel('correct')
    ax.legend(loc='lower right', frameon=False)
    plt.show()
    print(f'Trained on only {per_class} drawings per class: {tr[-1]:.0%} on those, {te[-1]:.0%} on new ones. '
          f'"95% accurate" means nothing until you ask: on what?')

print('✓ Setup done.' + ('' if IN_COLAB else ' (Not in Colab: drawing will fall back to random test doodles.)'))

## 1 · Get the data

We download 5,000 drawings for each of five categories, made by real people playing Google's [Quick, Draw!](https://quickdraw.withgoogle.com/data) game. Four thousand of each are for **training**. The other thousand are **held out** so we can test the network on drawings it has never seen.

In [ ]:
data = get_data()
print(f'{len(data.train.y):,} training drawings, {len(data.test.y):,} held-out drawings')
show_examples(data)

A drawing is just a 28 × 28 grid of numbers from 0 (black) to 255 (white). That is **all** the network ever gets. It has no idea it's looking at a cat.

In [ ]:
show_as_numbers(data)

## 2 · Build and train the network

The network has three layers. Each is a pile of **weights**: adjustable numbers, like the knobs in the water demo.

In [ ]:
model = TinyCNN()
describe(model)

This is the entire training loop. Each line is one step from the water demo: pour, compare, work backward, turn the knobs. Then repeat thousands of times.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
plot = LossPlot()

for epoch in range(3):                                      # 3 passes over all 20,000 training drawings
    for images, labels in batches(data.train, size=128):
        predictions = model(images)                         # 1. forward pass: make a guess
        loss = F.cross_entropy(predictions, labels)         # 2. loss: how wrong was the guess?
        optimizer.zero_grad()
        loss.backward()                                     # 3. backpropagation: how did each weight affect the loss?
        optimizer.step()                                    # 4. optimizer: nudge every weight to lower the loss
        plot.add(loss.item())

plot.done()
report(model, data)

## 3 · Look inside

### What did layer 1 learn?
Each of the 8 filters is a 3 × 3 grid of weights. **Nobody designed these.** They started as random numbers and training shaped them. Most end up detecting simple things: an edge, a stroke in one direction, a dark-to-light boundary.

In [ ]:
show_filters(model)

### What happens to one drawing, layer by layer
Each filter slides across the image and lights up wherever its pattern appears. Layer 1 finds strokes and edges. Layer 2 combines *those* into larger parts at half the resolution. The last layer turns everything into one score per class.

Change `'cat'` to `'fish'`, `'house'`, `'tree'` or `'bicycle'`, or change the `0` to see a different drawing.

In [ ]:
show_layers(model, pick(data, 'cat', 0))

### What does each layer-2 channel look for?
For eight of the layer-2 channels, these are the held-out drawings that made the channel respond most strongly. The red box shows where. Some channels have clearly specialized: round wheels, pointy roofs, tails. Nobody told them to.

In [ ]:
show_channel_favorites(model, data)

## 4 · Draw your own

Run the cell and draw a **cat, fish, house, tree or bicycle**. Draw big, then click **Done**. Run it again to draw something else.

Try something that is *none* of those, too. What does it say?

In [ ]:
my_drawing = draw()
show_layers(model, my_drawing)

## 5 · How layers bend space

Why use many layers? Here's a network small enough to watch: two tangled groups of points, and hidden layers with only **2 neurons each**, so every layer can be plotted on flat paper.

Watch the grey grid. Each layer **stretches and folds** the space a little. By the last layer the two groups have been pulled apart so far that a single straight line separates them. Deep networks do the same thing in thousands of dimensions.

In [ ]:
bend_space_demo()

## 6 · When it goes wrong

### a) Skewed training data
We retrain the exact same network, but give it only 5% of the fish drawings. Nothing about the code changes. Only the data does.

In [ ]:
skewed_data_demo(model, data, rare='fish', keep=0.05)

Real systems inherit the same problem. If some group is rare in the training data, a model can quietly be worse for that group, and the overall accuracy number can hide it.

### b) Something it has never seen
It only knows five answers. What happens when we show it bananas?

In [ ]:
never_seen_demo(model, 'banana')

The network can't say *"I don't know."* It spreads probability over the answers it has, often confidently. Language models have a version of this: they always produce a likely-sounding next word, whether or not the facts are there.

### c) Memorizing vs. generalizing
Train on only 40 drawings per class, over and over.

In [ ]:
memorize_demo(data, per_class=40)

It gets nearly every *training* drawing right by memorizing them, while doing far worse on new drawings. That gap is **overfitting**, and it's why honest accuracy is always measured on held-out data.

---

## Takeaways

- A network only ever sees **numbers**.
- **Training** is the loop: forward pass → loss → backpropagation → optimizer step, repeated thousands of times.
- Nobody writes the features. **Layers learn them**: edges first, then parts, then a decision.
- Depth lets a network **bend space** until hard problems become simple ones.
- What it learns depends entirely on the **data**. Skewed data gives skewed results, and it will answer confidently even outside what it knows.